# 분기 비교: IQR 이상치 제거 있음 vs 없음

`build_with_iqr.py` / `build_without_iqr.py`로 만든 두 데이터셋을 비교합니다.
이 노트북은 **데이터 자체의 차이**만 봅니다 (몇 개 값이 결측 처리됐는지, 분포가 어떻게 바뀌는지).

**아직 GPU 학습 결과 비교는 아닙니다** -- 어떤 게 최종적으로 더 나은 모델을 만드는지는
두 데이터셋 각각으로 학습(`train/finetune_chronos2.py`) + 백테스트(`eval/backtest.py`)까지
돌린 뒤에만 판단할 수 있습니다. 이 노트북 맨 아래 '5. 결론' 셀에 그 결과까지 정리해서 최종
선택을 적어주세요.

먼저 GPU 머신(원본 xlsx가 있는 컴퓨터)에서 아래 두 개를 실행해서 두 데이터셋을 만들어야 합니다:
```
python notebooks/01_iqr_outlier_removal/build_with_iqr.py
python notebooks/01_iqr_outlier_removal/build_without_iqr.py
```

커널: **Python (chronos_quality_net)** 선택 후 실행하세요.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
matplotlib.rcParams['axes.unicode_minus'] = False

# 이 노트북은 notebooks/01_iqr_outlier_removal/ 안에서 실행된다고 가정
ROOT = Path.cwd()
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError(
            "config.py를 찾지 못했습니다. Jupyter를 chronos_quality_net 폴더(또는 그 하위)에서 "
            "실행하거나, 이 노트북을 그 폴더 밑으로 옮겨서 다시 여세요."
        )
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import config as cfg

WITH_IQR_PATH = ROOT / "data" / "processed" / "with_iqr" / "quality_timeseries.csv"
WITHOUT_IQR_PATH = ROOT / "data" / "processed" / "without_iqr" / "quality_timeseries.csv"

for p in (WITH_IQR_PATH, WITHOUT_IQR_PATH):
    if not p.exists():
        print(f"missing: {p}  <- 먼저 build_with_iqr.py / build_without_iqr.py를 GPU 머신에서 실행하세요.")

print("targets:", cfg.TARGET_COLS)

## 1. 두 데이터셋 불러오기

In [ ]:
df_with = pd.read_csv(WITH_IQR_PATH, parse_dates=["timestamp"])
df_without = pd.read_csv(WITHOUT_IQR_PATH, parse_dates=["timestamp"])

assert list(df_with.columns) == list(df_without.columns), "두 데이터셋의 컬럼이 다릅니다 -- prepare_base()/finalize_and_save() 버전이 서로 다를 수 있습니다."
assert df_with.shape == df_without.shape, "두 데이터셋의 행/열 개수가 다릅니다 -- IQR은 값만 NaN으로 바꿔야지 행을 지우면 안 됩니다."

print(f"with_iqr:    {df_with.shape}")
print(f"without_iqr: {df_without.shape}")
print("컬럼/행 개수 동일함 확인 완료 (IQR은 값만 NaN 처리, 행 삭제 없음이 맞음)")

## 2. IQR로 실제로 몇 개 값이 제거됐는지 (컬럼별)

`without_iqr`에는 값이 있는데 `with_iqr`에는 NaN인 셀 = IQR이 이상치로 판단해서 지운 값.

In [ ]:
feature_cols = cfg.CONTROL_COLS + cfg.MONITOR_COLS + cfg.TARGET_COLS

removed_counts = {}
for col in feature_cols:
    removed = (df_without[col].notna() & df_with[col].isna()).sum()
    removed_counts[col] = int(removed)

removed_series = pd.Series(removed_counts).sort_values(ascending=False)
print(f"총 IQR로 제거된 값: {removed_series.sum()}")
removed_series[removed_series > 0].to_frame("removed_by_iqr")

## 3. 타깃(blaine/residue) 통계 비교

In [ ]:
rows = []
for target in cfg.TARGET_COLS:
    for label, df in [("with_iqr", df_with), ("without_iqr", df_without)]:
        s = df[target].dropna()
        rows.append({
            "target": target, "variant": label, "n": len(s),
            "mean": s.mean(), "std": s.std(), "min": s.min(), "max": s.max(),
        })
pd.DataFrame(rows).set_index(["target", "variant"]).round(3)

## 4. 분포 비교 (박스플롯)

In [ ]:
fig, axes = plt.subplots(1, len(cfg.TARGET_COLS), figsize=(5 * len(cfg.TARGET_COLS), 4.5))
if len(cfg.TARGET_COLS) == 1:
    axes = [axes]

for ax, target in zip(axes, cfg.TARGET_COLS):
    data = [df_with[target].dropna(), df_without[target].dropna()]
    ax.boxplot(data, labels=["with_iqr", "without_iqr"])
    lo, hi = cfg.SPEC_RANGES[target]
    ax.axhspan(lo, hi, color="green", alpha=0.06, label="spec 범위")
    ax.set_title(target)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. 결론 (직접 채워주세요)

**데이터 차이:**
- (위 표/그래프 보고 요약)

**모델 성능 비교 (GPU 머신에서 학습/백테스트 후 채우기):**
```
python train/finetune_chronos2.py --target blaine ...   # with_iqr, without_iqr 각각
python eval/backtest.py --target blaine ...              # 마찬가지로 각각
```
| variant | target | MAE | RMSE | R2 | spec accuracy |
|---|---|---|---|---|---|
| with_iqr | blaine | | | | |
| without_iqr | blaine | | | | |
| with_iqr | residue | | | | |
| without_iqr | residue | | | | |

**최종 선택:** (with_iqr / without_iqr) -- 이유: 